# M05-02 — Acumulados por entidad

[← Anterior](02-lab-ranking-ventana.ipynb) · [Siguiente →](../M06-optimizacion-ejecucion/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Numerar los pedidos de cada cliente y calcular el GMV cobrable acumulado en el tiempo.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M05-02-acumulados.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Grano pedido (no línea)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Un pedido con 3 líneas no es 3 visitas. Agrego a order_id.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** **469** pedidos cobrables con cliente (el mismo count que el KPI de M04-02).

**Por qué este paso.** Si te salen 1122, no agregaste a order_id.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, min as fmin, sum as fsum, row_number

spark = get_spark("novashop-m05")
orders_gmv = (
    spark.read.parquet(str(STAGING / "fact_lines"))
    .join(spark.read.parquet(str(STAGING / "customers_clean")), "customer_id", "inner")
    .where(col("is_billable"))
    .groupBy("customer_id", "order_id")
    .agg(
        fmin("order_ts").alias("order_ts"),
        fsum("gmv_line").alias("gmv"),
    )
)
print(orders_gmv.count())


### Paso 2 — Número de pedido y acumulado

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

La misma window sirve para el índice y para el sum. El orderBy de la window ES el tiempo.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `order_n` 1, 2, 3… por cliente; `gmv_running` no decrece dentro del mismo customer_id.

**Por qué este paso.** Si baja, el orderBy de la window no es order_ts.


In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy("order_ts")
hist = (
    orders_gmv.withColumn("order_n", row_number().over(w))
    .withColumn("gmv_running", fsum("gmv").over(w))
)
hist.orderBy("customer_id", "order_n").show(12)


### Paso 3 — Primera compra vs repetición

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

order_n == 1 es la definición operativa de “nuevos” en este curso.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** ~211 primeras compras (un true por cliente con paid) y el resto repeticiones.

**Por qué este paso.** Sin partitionBy el acumulado es de toda la empresa.


In [ ]:
hist.groupBy((col("order_n") == 1).alias("is_first")).count().show()


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

En un cliente con `order_n` ≥ 2, `gmv_running` de la fila 2 ≥ fila 1.
Si baja, corrige el orderBy. Déjalo escrito en Markdown.


## Mejora — Pedidos hasta superar 1000 €

Quédate, por cliente, con la primera fila donde `gmv_running >= 1000` (o ninguna).

Si te atasca, el código está en la celda siguiente.


In [ ]:
w2 = Window.partitionBy("customer_id").orderBy("order_ts")
crossed = hist.where(col("gmv_running") >= 1000)
first_cross = crossed.withColumn("rn", row_number().over(w2)).where(col("rn") == 1)
first_cross.select("customer_id", "order_n", "gmv_running").show()


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| gmv_running igual en todas las filas | Window sin orderBy | partitionBy + orderBy(order_ts) |
| 1122 “pedidos” | No agregaste a order_id | Paso 1 |
| Acumulado a nivel empresa | Falta partitionBy | Añádelo |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M06 — teoría](../M06-optimizacion-ejecucion/01-teoria.ipynb).
